# YOLOv8s × VisDrone2019 Fine-Tuning
**Derin Öğrenme ile Güvenlik İhlali Tespiti**

Bu notebook Google Colab T4/A100 GPU üzerinde çalıştırılmak üzere hazırlanmıştır.

Akış:
1. GPU kontrol
2. Bağımlılık kurulumu
3. VisDrone2019 dataset indirme
4. YOLO formatına dönüştürme
5. Fine-tuning (80 epoch)
6. Modeli Drive'a kaydetme

## 1 — GPU Kontrolü

In [ ]:
import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU :', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('GPU bulunamadı — Colab Runtime > Change runtime type > T4 GPU seçin')

## 2 — Bağımlılıklar

In [ ]:
!pip install -q ultralytics albumentations opencv-python-headless pyyaml tqdm

## 3 — Proje Reposu

In [ ]:
import os

REPO_URL = 'https://github.com/EmineCakal5/Ada_Project.git'
WORK_DIR = '/content/Ada_Project'

if not os.path.exists(WORK_DIR):
    !git clone {REPO_URL} {WORK_DIR}
else:
    !git -C {WORK_DIR} pull

os.chdir(WORK_DIR)
print('Çalışma dizini:', os.getcwd())

## 4 — VisDrone2019 Dataset İndirme

İki seçenek:
- **A) Kaggle API** (önerilen — tam dataset ~2.5 GB)
- **B) Google Drive** (kendi Drive'ınıza önceden yüklediyseniz)

Sadece birini çalıştırın.

In [ ]:
# ── SEÇENEK A: Kaggle API ──────────────────────────────────────
# kaggle.json dosyanızı Colab'a yükleyin (Files > Upload)
# ya da aşağıya username/key girin

import os, json

KAGGLE_USERNAME = ''   # <- buraya Kaggle kullanıcı adınızı yazın
KAGGLE_KEY      = ''   # <- buraya Kaggle API key'inizi yazın

if KAGGLE_USERNAME and KAGGLE_KEY:
    os.makedirs('/root/.kaggle', exist_ok=True)
    with open('/root/.kaggle/kaggle.json', 'w') as f:
        json.dump({'username': KAGGLE_USERNAME, 'key': KAGGLE_KEY}, f)
    os.chmod('/root/.kaggle/kaggle.json', 0o600)
    !pip install -q kaggle
    !kaggle datasets download -d ultralytics/visdrone -p /content/ --unzip
    VISDRONE_ROOT = '/content/visdrone'
    print('VisDrone indirildi:', VISDRONE_ROOT)
else:
    print('Kaggle bilgileri girilmedi — Seçenek B\'yi deneyin.')

In [ ]:
# ── SEÇENEK B: Google Drive ────────────────────────────────────
# Drive'ınızdaki VisDrone2019 klasörünün yolunu yazın

from google.colab import drive
drive.mount('/content/drive')

# Drive'daki VisDrone2019 kök klasörünün yolu:
VISDRONE_ROOT = '/content/drive/MyDrive/VisDrone2019'  # <- güncelleyin

import os
if os.path.exists(VISDRONE_ROOT):
    print('VisDrone bulundu:', VISDRONE_ROOT)
    print(os.listdir(VISDRONE_ROOT))
else:
    print('Hata: Klasör bulunamadı →', VISDRONE_ROOT)

## 5 — YOLO Formatına Dönüştürme

In [ ]:
import subprocess, sys

DST = '/content/Ada_Project/data/visdrone'

result = subprocess.run(
    [sys.executable, 'tools/prepare_visdrone.py',
     '--src', VISDRONE_ROOT,
     '--dst', DST],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print('HATA:', result.stderr)

In [ ]:
# Dönüşüm özeti
import os
for split in ['train', 'val', 'test']:
    img_count = len(os.listdir(f'{DST}/images/{split}')) if os.path.exists(f'{DST}/images/{split}') else 0
    print(f'{split:5s}: {img_count} görüntü')

## 6 — Fine-Tuning

T4 GPU → ~2–3 saat (80 epoch)  
A100 GPU → ~40–50 dakika (80 epoch)

In [ ]:
import subprocess, sys, torch

DEVICE = '0' if torch.cuda.is_available() else 'cpu'
EPOCHS = 80
IMGSZ  = 960   # VisDrone küçük nesneler için 960 önerilir
BATCH  = 8     # T4 (16 GB): 8-12 güvenli; A100: 16-32

result = subprocess.run(
    [sys.executable, 'tools/train_drone_yolo.py',
     '--data',    'data/visdrone/visdrone.yaml',
     '--epochs',  str(EPOCHS),
     '--imgsz',   str(IMGSZ),
     '--batch',   str(BATCH),
     '--device',  DEVICE,
     '--out',     'models/weights/yolov8s_drone.pt'],
    # Çıktıyı canlı göster
    stdout=None, stderr=None
)
print('Çıkış kodu:', result.returncode)

## 7 — Sonuçlar

In [ ]:
import os
from IPython.display import Image, display

model_path = 'models/weights/yolov8s_drone.pt'
if os.path.exists(model_path):
    size_mb = os.path.getsize(model_path) / 1e6
    print(f'Model kaydedildi: {model_path} ({size_mb:.1f} MB)')
else:
    print('Model bulunamadı — eğitim hatası olabilir.')

# Ultralytics eğitim grafiği
results_img = 'runs/drone/yolov8s_visdrone/results.png'
if os.path.exists(results_img):
    display(Image(results_img))

## 8 — Modeli Drive'a Kaydet

In [ ]:
import shutil, os
from google.colab import drive

# Drive henüz mount edilmediyse:
# drive.mount('/content/drive')

SAVE_DIR = '/content/drive/MyDrive/Ada_Project_Models'
os.makedirs(SAVE_DIR, exist_ok=True)

src = 'models/weights/yolov8s_drone.pt'
dst = os.path.join(SAVE_DIR, 'yolov8s_drone.pt')

if os.path.exists(src):
    shutil.copy2(src, dst)
    print('Kaydedildi →', dst)
else:
    print('Kaynak model bulunamadı.')

## 9 — Modeli Doğrula (İsteğe Bağlı)

In [ ]:
from ultralytics import YOLO

model = YOLO('models/weights/yolov8s_drone.pt')
metrics = model.val(data='data/visdrone/visdrone.yaml', imgsz=960, device=DEVICE)

print(f'mAP@0.5     : {metrics.box.map50:.3f}')
print(f'mAP@0.5:0.95: {metrics.box.map:.3f}')
print(f'Precision   : {metrics.box.mp:.3f}')
print(f'Recall      : {metrics.box.mr:.3f}')

---
## Eğitim Sonrası

Modeli indirip projeye eklemek için:

```bash
# yolov8s_drone.pt dosyasını Ada_Project/models/weights/ klasörüne koyun
# ardından config.yaml'da:
# detector:
#   model: "models/weights/yolov8s_drone.pt"
```